<a href="https://colab.research.google.com/github/ounospanas/AIDL_B_CS01/blob/main/Retrieving_Similar_News_Posts_with_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieving similar news posts with Transformers

## Goal of this lab
We have a collection of about **200,000 news posts** (HuffPost, 2012–2022). A user writes a **new post**, for example *"A man killed his wife"*.
We want to find the news posts in the collection that are **most similar in meaning** to it.

We do it in **two stages**, the standard design of modern search engines:

```
STAGE 1 – fast, rough  (bi-encoder = Sentence-RoBERTa)
   new post ──► vector ──┐
                         ├──► cosine similarity with all 200k stored vectors ──► top 32 candidates
   200k posts ─► vectors ┘    (the 200k vectors are computed only once and saved)

STAGE 2 – slow, accurate  (cross-encoder = our fine-tuned RoBERTa)
   (new post, candidate 1)  ──► RoBERTa ──► similarity score 0–5
   (new post, candidate 2)  ──► RoBERTa ──► similarity score 0–5
   ...  32 pairs only  ...                 ──► re-ranked list, best match first
```

## What you will learn
1. **Fine-tune** a pre-trained Transformer (RoBERTa-large) on a **regression** task: STS-B (how similar are two sentences, 0–5).
2. Download a real dataset from **Kaggle**.
3. Turn sentences into **sentence embeddings** with the `sentence-transformers` library.
4. **Retrieve** the most similar posts with **cosine similarity**.
5. **Re-rank** the candidates with the fine-tuned model and see why the two-stage design is used.

## How to run it
- Run the cells **in order**, from top to bottom: click a cell and press **Shift + Enter**.
- You need a **GPU** (Colab: *Runtime → Change runtime type → GPU*).
- Two steps are slow: training (Part 1, about 15 min) and encoding all posts (Part 3, about 20 min on a T4). Times on other GPUs differ.
- If you restart the kernel, run the cells again from the top. Every variable lives only in memory.

## Mini glossary
| Term | Meaning |
|---|---|
| **Transformer** | Neural network built from self-attention layers (BERT, RoBERTa, GPT…). |
| **RoBERTa** | A BERT-like encoder, trained longer and on more text than BERT. `roberta-large` has 24 layers and 355M parameters. |
| **Checkpoint** | The name of a saved, pre-trained model on the Hugging Face Hub, e.g. `FacebookAI/roberta-large`. |
| **Tokenizer** | Turns text into token IDs (numbers) the model understands, and back. |
| **Fine-tuning** | Continuing the training of a pre-trained model on our own task, with a small learning rate. |
| **Regression** | The model predicts **one real number** (here a score 0–5), not a class. |
| **Epoch / batch** | Epoch = one full pass over the training data. Batch = the small group of examples processed together (here 8). |
| **DataLoader** | PyTorch object that cuts the dataset into batches and shuffles it. |
| **Optimizer / scheduler** | Optimizer = the rule that updates the weights (AdamW). Scheduler = changes the learning rate during training. |
| **Sentence embedding** | One vector (here 1024 numbers) that represents the meaning of a whole sentence. |
| **Cosine similarity** | Measures the angle between two vectors: 1 = same direction (similar meaning), 0 = unrelated. |
| **Bi-encoder** | Encodes each text **alone** into a vector; compare vectors afterwards. Fast, vectors can be stored. |
| **Cross-encoder** | Reads **both texts together** and outputs one score. Accurate, but must run once per pair. |
| **Re-ranking** | Re-ordering a short candidate list with a more accurate (slower) model. |
| **Pearson / Spearman** | Correlation between predictions and true scores. Pearson: linear agreement of the values. Spearman: agreement of the ranking order. 1 = perfect. |

# Part 1 – Fine-tune RoBERTa on STS-B (our cross-encoder)

### Summary
**STS-B** (Semantic Textual Similarity Benchmark) contains **sentence pairs** with a human score from **0** (unrelated) to **5** (same meaning):

| sentence1 | sentence2 | label |
|---|---|---|
| A plane is taking off. | An air plane is taking off. | 5.0 |
| A man is playing a large flute. | A man is playing a flute. | 3.8 |
| A woman is dancing. | A man is talking. | 0.0 *(made-up example)* |

Splits: **5,749 train**, **1,500 validation**, 1,379 test (the test labels are hidden, so we evaluate on validation).

We give RoBERTa **both sentences at once**. The tokenizer joins them with special tokens:

```
<s> A plane is taking off. </s></s> An air plane is taking off. </s>
```

Because the two sentences are inside the same input, every word of sentence 1 can attend to every word of sentence 2 in every layer. That is what makes it a **cross-encoder**.

On top of RoBERTa we put a small "head" with **one output** (`num_labels=1`), so the model predicts one number. With one output, Hugging Face automatically uses **regression** with the **MSE loss** (mean squared error).

In [1]:
# Install the libraries (they may already be installed; then pip just says "Requirement already satisfied").
# %pip (instead of !pip) installs into the same Python environment the notebook kernel is using.
%pip install -q datasets "transformers[sentencepiece]" sentence-transformers scipy scikit-learn pandas


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Load the data and look at one example

In [2]:
from datasets import load_dataset          # Hugging Face library that downloads ready-made datasets
from transformers import set_seed          # helper that fixes all random generators

set_seed(42)                               # same shuffling and same head initialisation in every run

# Download STS-B. It is one of the GLUE benchmark tasks, stored in the "nyu-mll/glue" repository.
# "stsb" selects the STS-B part of GLUE.
raw_datasets = load_dataset("nyu-mll/glue", "stsb")

print(raw_datasets)                        # shows the three splits and their columns
print(raw_datasets["train"][0])            # first training example: two sentences, a score, an index
# Expected: train 5749 rows, validation 1500, test 1379
#           columns: sentence1, sentence2, label, idx

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})
{'sentence1': 'A plane is taking off.', 'sentence2': 'An air plane is taking off.', 'label': 5.0, 'idx': 0}


### Tokenize the sentence pairs

In [3]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# The pre-trained model we start from. "FacebookAI/roberta-large" is the official name of roberta-large.
checkpoint = "FacebookAI/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)   # download the matching tokenizer

# Tokenization function: receives a batch of examples (a dict of lists)
def tokenize_function(example):
    # Passing two texts makes the tokenizer build ONE input: <s> sentence1 </s></s> sentence2 </s>
    # truncation=True cuts inputs longer than the model maximum (512 tokens). STS-B pairs are much shorter.
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

# Apply the function to every split. batched=True sends 1000 examples at a time (much faster).
# New columns are added: input_ids (token numbers) and attention_mask (1 = real token).
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# The collator pads each batch to the length of its LONGEST pair ("dynamic padding"),
# not to 512, so batches stay small and fast.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Look at what the model will really read for the first pair
print(tokenizer.decode(tokenized_datasets["train"][0]["input_ids"]))
# Expected: <s>A plane is taking off.</s></s>An air plane is taking off.</s>

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

<s>A plane is taking off.</s></s>An air plane is taking off.</s>


In [4]:
# Keep only the columns the model needs.
# The model cannot use raw text or the example index, so we remove them.
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])

# Hugging Face models expect the target in a column called "labels" (plural).
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Return PyTorch tensors instead of Python lists when we read examples.
tokenized_datasets.set_format("torch")

tokenized_datasets["train"].column_names
# Expected: ['labels', 'input_ids', 'attention_mask']

['labels', 'input_ids', 'attention_mask']

### Batches
A **DataLoader** cuts the dataset into batches of 8 pairs. For training it also **shuffles** the order in every epoch, so the model doesn't learn the order of the file. Validation doesn't need shuffling.

In [5]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,               # new random order in every epoch
    batch_size=8,               # 8 sentence pairs per step
    collate_fn=data_collator,   # pads each batch to its longest pair
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=8,
    collate_fn=data_collator,
)

# Look inside one batch: every tensor has 8 rows (one per pair)
batch = next(iter(train_dataloader))
{k: v.shape for k, v in batch.items()}
# Expected (the second number changes from batch to batch, it is the longest pair in that batch):
# {'labels': torch.Size([8]), 'input_ids': torch.Size([8, 30]), 'attention_mask': torch.Size([8, 30])}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 39]),
 'attention_mask': torch.Size([8, 39])}

### The model
`AutoModelForSequenceClassification` = RoBERTa body + a small head on the first token `<s>` (RoBERTa's equivalent of BERT's `[CLS]`).
With `num_labels=1` the head outputs one number, and the loss becomes MSE (regression).

You will see a warning that some weights were **not used** (the pre-training head `lm_head…`) and some are **newly initialized** (`classifier…`). That is expected: we throw away the word-prediction head and train a new one-number head.

In [6]:
from transformers import AutoModelForSequenceClassification

# Load pre-trained RoBERTa-large and put a new head with ONE output on top (regression task)
# (model.config.problem_type is set to "regression" automatically at the first forward pass with labels)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Optimizer and learning-rate schedule
- **AdamW**: the usual optimizer for Transformers. `lr=1e-5` is small on purpose: the pre-trained weights are already good, we only want to adjust them gently.
- **Linear scheduler**: the learning rate goes down in a straight line from 1e-5 to 0 by the last step.
- Steps = epochs × batches per epoch = 3 × 719 = **2157** (5,749 / 8 = 718.6, rounded up).

In [7]:
import torch
from transformers import get_scheduler

# AdamW now comes from PyTorch (the copy inside transformers was removed)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
num_epochs = 3

# Total number of weight updates in the whole training
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear",                           # learning rate decreases linearly to 0
    optimizer=optimizer,
    num_warmup_steps=0,                 # no warm-up phase
    num_training_steps=num_training_steps,
)
print(num_training_steps)               # Expected: 2157

2157


In [8]:
# Use the GPU if there is one, otherwise the CPU (training on CPU would take many hours)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)                        # copy all model weights to that device
device
# Expected: device(type='cuda')

device(type='cuda')

### Training loop
For every batch the same four steps repeat:
1. **forward**: the model predicts a score for each pair and, because we pass `labels`, also computes the MSE loss;
2. **backward**: PyTorch computes the gradient of the loss for every weight;
3. **step**: the optimizer changes the weights a little, and the scheduler lowers the learning rate;
4. **zero_grad**: we clear the gradients, otherwise PyTorch would add them to the next batch's gradients.

In [9]:
# Training loop (about 15 minutes on a Colab T4)
from tqdm.auto import tqdm                       # progress bar

progress_bar = tqdm(range(num_training_steps))

model.train()                                    # training mode: dropout is active
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in train_dataloader:
        # move every tensor of the batch (input_ids, attention_mask, labels) to the GPU
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)                 # forward pass; ** unpacks the dict into keyword arguments
        loss = outputs.loss                      # MSE between predicted and true scores
        loss.backward()                          # backward pass: compute gradients

        optimizer.step()                         # update the weights
        lr_scheduler.step()                      # lower the learning rate a little
        optimizer.zero_grad()                    # clear gradients for the next batch
        progress_bar.update(1)
        epoch_loss += loss.item()                # .item() turns a 1-number tensor into a Python float

    print(f"epoch {epoch + 1}: mean training loss = {epoch_loss / len(train_dataloader):.3f}")
# The loss should go down from epoch to epoch.

  0%|          | 0/2157 [00:00<?, ?it/s]

epoch 1: mean training loss = 1.111
epoch 2: mean training loss = 0.352
epoch 3: mean training loss = 0.247


### Evaluation on the validation set
STS-B is evaluated with **correlations** between the predicted and the true scores:
- **Pearson**: are the predicted *values* linearly related to the true ones?
- **Spearman**: is the *ranking order* the same (the most similar pair predicted highest, etc.)?

In [32]:
# Run inference on the validation set and compute the STS-B scores
from scipy.stats import pearsonr, spearmanr

model.eval()                                       # evaluation mode: dropout off
all_preds, all_labels = [], []
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():                          # no gradients needed: faster, less memory
        outputs = model(**batch)

    # logits has shape (8, 1): one column. squeeze(-1) makes it (8,), like the labels.
    # .cpu() brings the numbers back from the GPU so NumPy/SciPy can use them.
    all_preds.extend(outputs.logits.squeeze(-1).cpu().tolist())
    all_labels.extend(batch["labels"].cpu().tolist())

print({"pearson": pearsonr(all_preds, all_labels)[0],
       "spearmanr": spearmanr(all_preds, all_labels)[0]})
# Original run: {'pearson': 0.924, 'spearmanr': 0.922}. Expect about 0.92, a little different in every run.

{'pearson': 0.9181003808796372, 'spearmanr': 0.916424375391885}


In [11]:
# Save the fine-tuned weights (about 1.4 GB file), so we don't have to train again
torch.save(model.state_dict(), "roberta_stsb.pt")

In [12]:
# Load the saved weights back into the model.
# After a kernel restart: first run the cells that define checkpoint, tokenizer, model and device,
# then this cell, and you can skip the training loop.
model.load_state_dict(torch.load("roberta_stsb.pt", map_location=device))
# Expected: <All keys matched successfully>

<All keys matched successfully>

In [13]:
# Free GPU memory before Part 3.
# The optimizer keeps two extra numbers per weight (about 2.8 GB for RoBERTa-large),
# and Part 3 loads a SECOND RoBERTa-large (the sentence embedder). On a small GPU or a GPU slice
# this cleanup avoids an out-of-memory error. The fine-tuned model itself stays: we need it in Part 5.
del optimizer, lr_scheduler
model.zero_grad(set_to_none=True)       # drop the stored gradients
if torch.cuda.is_available():
    torch.cuda.empty_cache()            # give the freed memory back
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

GPU memory in use: 1.4 GB


# Part 2 – Download the example dataset (News Category Dataset)

### Summary
The **News Category Dataset** (Kaggle, `rmisra/news-category-dataset`) has about **210,000 HuffPost news** from 2012–2022.
Each news item has: `link`, `headline`, `category`, `short_description`, `authors`, `date`. We use `short_description`.

To download from Kaggle with code you need an **API token**, the file `kaggle.json`
(Kaggle → your profile → *Settings* → *API* → *Create New Token*).

⚠️ `kaggle.json` contains your **secret key**. Never print it, never share a notebook whose output shows it,
never upload it to GitHub. If it was exposed, expire it in the same Kaggle page and create a new one.

**Three ways to get the data** (choose one):
1. **Colab**: the next cells ask you to upload `kaggle.json`.
2. **Local Jupyter**: put `kaggle.json` in `~/.kaggle/` yourself (Linux/Mac; on Windows `C:\Users\<you>\.kaggle\`).
3. **No API**: download the zip from https://www.kaggle.com/datasets/rmisra/news-category-dataset in the browser
   and unzip it so that the file `news/News_Category_Dataset_v3.json` exists next to this notebook. The download cell then does nothing.

In [14]:
# Install the Kaggle command-line tool
%pip install -q kaggle


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os, shutil

# Are we in Google Colab? The module google.colab exists only there.
try:
    from google.colab import files       # Colab helper for uploading files from your computer
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

Running in Colab: False


In [16]:
# Colab only: upload kaggle.json and put it where the Kaggle tool looks for it (~/.kaggle/)
#
#kaggle_dir = os.path.expanduser("~/.kaggle")       # "~" = your home folder
#
#if IN_COLAB and not os.path.exists(DATA_FILE):
#    # We store the result in a variable. If files.upload() is the LAST line of a cell,
#    # Jupyter prints its result, and that result is the file content: your secret key!
#    uploaded = files.upload()                      # opens a "Choose file" button
#    os.makedirs(kaggle_dir, exist_ok=True)         # exist_ok: no error if the folder already exists
#    shutil.copy("kaggle.json", kaggle_dir)
#    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)   # only you may read the key file
#    print("kaggle.json installed")

############################Modified Code to Word with any Jupyter Lab notebook######################
# Put kaggle.json where the Kaggle tool looks for it (~/.kaggle/).
# Colab: you upload it with a button. Local Jupyter: copy it next to this notebook first.
DATA_FILE = "news/News_Category_Dataset_v3.json"
kaggle_dir = os.path.expanduser("~/.kaggle")          # "~" = your home folder
kaggle_key = os.path.join(kaggle_dir, "kaggle.json")

if not os.path.exists(DATA_FILE) and not os.path.exists(kaggle_key):
    if IN_COLAB:
        # Stored in a variable on purpose: if files.upload() is the LAST line of a cell,
        # Jupyter prints its result, which is the file content, i.e. your secret key!
        uploaded = files.upload()                     # opens a "Choose file" button
    # Now kaggle.json is in the notebook folder (uploaded in Colab, or put there by you)
    os.makedirs(kaggle_dir, exist_ok=True)            # exist_ok: no error if the folder exists
    shutil.copy("kaggle.json", kaggle_key)
    os.chmod(kaggle_key, 0o600)                       # only you may read the key file
    print("kaggle.json installed in", kaggle_dir)

In [17]:
# Download the news dataset and unzip it into the folder "news" (skipped if the file is already there)
import subprocess

if os.path.exists(DATA_FILE):
    print("Dataset already present:", DATA_FILE)
else:
    # Same as typing in a terminal:  kaggle datasets download rmisra/news-category-dataset -p news --unzip
    subprocess.run(["kaggle", "datasets", "download", "rmisra/news-category-dataset",
                    "-p", "news", "--unzip"], check=True)   # check=True: stop with an error if it fails
print(os.listdir("news"))
# Expected: ['News_Category_Dataset_v3.json']

Dataset URL: https://www.kaggle.com/datasets/rmisra/news-category-dataset
License(s): Attribution 4.0 International (CC BY 4.0)


100%|██████████| 26.5M/26.5M [00:01<00:00, 16.8MB/s]



['News_Category_Dataset_v3.json']


### Read the file
The file is in **JSON Lines** format: every line is one complete JSON object (one news item), not one big JSON list.
So we read it line by line.

In [18]:
# Convert the JSON Lines file to a list of dictionaries
import json

list_ = []
with open(DATA_FILE) as f:          # f = the open file (named "files" in the original, which hid Colab's files)
    for line in f:                  # one line = one news item
        list_.append(json.loads(line))   # json.loads turns the text of one line into a Python dict

len(list_)
# Expected: about 210,000 (209527 in v3)

209527

In [19]:
# Convert the list of dicts to a pandas table (DataFrame): one row per news item, one column per key
import pandas as pd

data = pd.DataFrame(list_)
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


### Clean the descriptions
Thousands of news items have an **empty** `short_description`. An empty text still gets an embedding, which is noise in the search, so we remove these rows.

After removing rows we **reset the index** to 0, 1, 2, … . This is important: row *i* of `data` must be the same news item as embedding *i* in Part 3.

In [20]:
# Optional: use only part of the data for a quick class demo (encoding all posts takes ~20 min).
# None = use everything; e.g. 20000 = first 20,000 posts.
MAX_POSTS = None

n_before = len(data)
data = data[data["short_description"].str.strip() != ""]    # keep only non-empty descriptions
print(n_before - len(data), "empty descriptions removed")
if MAX_POSTS is not None:
    data = data.head(MAX_POSTS)
data = data.reset_index(drop=True)                          # new index 0..N-1, the old one is dropped

# The texts we will search in: a plain Python list of strings
descriptions = data["short_description"].tolist()
print(len(descriptions), "posts")
print(descriptions[0])

19712 empty descriptions removed
189815 posts
Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.


# Part 3 – Sentence embeddings with Sentence-Transformers (SRoBERTa, our bi-encoder)

### Summary
A **Sentence-Transformer** reads **one** sentence and returns **one vector**. Inside, it runs RoBERTa-large and takes the **average (mean pooling)** of all token vectors, which gives 1024 numbers per sentence.
It was trained (on NLI and STS-B data) so that sentences with similar meaning get vectors pointing in a similar direction.

Why not use our fine-tuned cross-encoder for everything? It needs **both** texts together. For one new post we would have to run RoBERTa-large 200,000 times. The bi-encoder encodes the 200,000 posts **once**, we save the vectors, and every new post then needs only **one** encoding plus a fast matrix comparison.

Note: `roberta-large-nli-stsb-mean-tokens` is the model from the original Sentence-BERT paper (2019). Its Hugging Face page now marks it as **deprecated**, because newer models give better embeddings. We keep it so the results match the lecture. **Try** `sentence-transformers/all-mpnet-base-v2` (better) or `sentence-transformers/all-MiniLM-L6-v2` (much faster): only `EMBEDDER_NAME` changes.

In [21]:
from sentence_transformers import SentenceTransformer

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
# Load the sentence embedder. The full name (with "sentence-transformers/") is required in recent versions.
EMBEDDER_NAME = "sentence-transformers/roberta-large-nli-stsb-mean-tokens"
embedder = SentenceTransformer(EMBEDDER_NAME, device=str(device))
print(embedder)        # shows the two parts: Transformer (RoBERTa) + Pooling (mean of the token vectors)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (6 outputs left out)

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'RobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
)


In [23]:
# Create the embeddings of all posts (about 20 min for all ~190k posts on a T4) and store them in a file.
# If the file exists from an earlier run and has the right number of rows, we just load it.
EMB_FILE = "post_embeddings.npy"

if os.path.exists(EMB_FILE) and np.load(EMB_FILE, mmap_mode="r").shape[0] == len(descriptions):
    post_embeddings = np.load(EMB_FILE)
    print("Loaded saved embeddings")
else:
    post_embeddings = embedder.encode(
        descriptions,
        batch_size=64,             # sentences per GPU batch (lower it if you get out-of-memory)
        show_progress_bar=True,
        convert_to_numpy=True,     # result is a NumPy matrix
    )
    np.save(EMB_FILE, post_embeddings)

post_embeddings.shape
# Expected: (number of posts, 1024)  →  one row of 1024 numbers per post

Batches:   0%|          | 0/2966 [00:00<?, ?it/s]

(189815, 1024)

# Part 4 – Retrieve the k most similar posts with cosine similarity (stage 1)

### Summary
**Cosine similarity** = the cosine of the angle between two vectors:

$$\cos(\mathbf{a},\mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}$$

1 = same direction (very similar meaning), 0 = unrelated, negative = opposite direction.
We compare the new post with **all** stored vectors at once (one matrix multiplication) and keep the **k = 32** best.

In [24]:
# An example new post, turned into a vector with the SAME embedder
input_post = "A man killed his wife"
input_emb = embedder.encode(input_post)
input_emb.shape
# Expected: (1024,)

(1024,)

In [25]:
%%timeit
# How long does one search over ALL stored posts take?
# %%timeit runs the cell several times and reports the average time.
# (Variables created inside a %%timeit cell are NOT kept afterwards.)

# [input_emb] = a list with one vector, i.e. a 1 × 1024 matrix; the result is a 1 × N matrix of similarities
cosine_similarity([input_emb], post_embeddings)
# Original run (Colab CPU, all posts): about 0.9 s per search

493 ms ± 130 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
# Function that returns the k most similar posts according to the bi-encoder (SRoBERTa)
def get_highest_similarity(embedding, post_embeddings, highest=32):
    """
    embedding:        vector of the new post, shape (1024,)
    post_embeddings:  matrix of all stored posts, shape (N, 1024)
    highest:          how many relevant posts to retrieve (k)
    returns:          dict {row index in `data`: cosine similarity}, best match first
    """
    # similarity of the new post with every stored post; [0] takes the single row → shape (N,)
    text_similarities = cosine_similarity([embedding], post_embeddings)[0]

    # np.argsort sorts from smallest to largest; with the minus sign we get largest first.
    # [:highest] keeps the k best positions (= row numbers in `data`).
    high_txt = np.argsort(-text_similarities)[:highest]

    # build the dict: row index → similarity (float() turns a NumPy number into a normal Python number)
    return {int(i): float(text_similarities[i]) for i in high_txt}

In [27]:
# Get the 32 most similar posts and show them, best first
highest_texts = get_highest_similarity(input_emb, post_embeddings, highest=32)

for rank, (i, sim) in enumerate(highest_texts.items(), start=1):
    print(f"{rank:2d}  cos={sim:.3f}  {data.iloc[i]['short_description'][:110]}")
# Original run: similarities between 0.58 and 0.69. The top hit was
#   "His first wife also died under suspicious circumstances while he was alone with her."
# and many hits are about killing or death in general (suicides, shootings),
# not specifically about a man killing his wife.

 1  cos=0.691  His first wife also died under suspicious circumstances while he was alone with her.
 2  cos=0.664  "I killed my son."
 3  cos=0.648  The 26-year-old internet star was strangled to death by her brother.
 4  cos=0.647  A California businessman is in custody on suspicion of murdering his wife, who has been missing since Wednesda
 5  cos=0.647  The shooter killed himself following the murders.
 6  cos=0.640  A man went into the meeting and killed his ex-girlfriend and her friend before shooting himself, police say.
 7  cos=0.636  The Paralympic gold medalist is currently in jail for killing his girlfriend.
 8  cos=0.632  Sgt. Michael Walker's girlfriend allegedly stabbed his wife to death in their bedroom.
 9  cos=0.630  The attacker then committed suicide.
10  cos=0.628  The shooter is also dead.
11  cos=0.625  The owner of the gun range was killed along with his wife and grandson.
12  cos=0.623  The man reportedly killed himself in front of law enforcement.
13  cos=0.616 

# Part 5 – Re-rank the candidates with the fine-tuned RoBERTa (stage 2)

### Summary
Now we use the **cross-encoder** from Part 1. For each of the 32 candidates we build the pair
`(new post, candidate)`, exactly in the format used in training (sentence1, sentence2), and the model predicts an **STS score (0–5)**.
We run the 32 pairs as **one batch**.

Why is this more accurate? In the bi-encoder the two texts never "see" each other: each is squeezed into one vector alone, and only the vectors are compared. In the cross-encoder, attention connects the words of the two texts in every layer ("man" ↔ "businessman", "killed" ↔ "murdering", "wife" ↔ "wife").
It is too slow for 200k posts, but 32 pairs take a fraction of a second.

In [28]:
# Build the two lists of the pairs: the new post repeated 32 times, and the 32 candidate descriptions
candidate_ids = list(highest_texts.keys())                       # row numbers in `data`
candidates = [data.iloc[i]["short_description"] for i in candidate_ids]
k_similar_posts = [[input_post, c] for c in candidates]          # list of pairs, used for printing below

# Tokenize as pairs, exactly as in training: tokenizer(sentence1_list, sentence2_list)
tokenized_similar_posts = tokenizer(
    [input_post] * len(candidates),   # sentence1 for every pair
    candidates,                       # sentence2 for every pair
    padding=True,                     # pad all 32 pairs to the same length (one rectangular batch)
    truncation=True,
    return_tensors="pt",              # return PyTorch tensors
)
print({k: v.shape for k, v in tokenized_similar_posts.items()})
# Expected: {'input_ids': torch.Size([32, L]), 'attention_mask': torch.Size([32, L])}, L = longest pair

tokenized_similar_posts = tokenized_similar_posts.to(device)    # move all tensors to the GPU

{'input_ids': torch.Size([32, 37]), 'attention_mask': torch.Size([32, 37])}


In [29]:
# Run inference with the fine-tuned RoBERTa-large STS-B model (one forward pass for all 32 pairs)
model.eval()
with torch.no_grad():
    outputs = model(**tokenized_similar_posts)

# One predicted STS score per pair: shape (32, 1) → (32,), then to NumPy
sts_scores = outputs.logits.squeeze(-1).cpu().numpy()
sts_scores.round(2)

array([2.38, 0.86, 1.58, 3.08, 1.06, 1.2 , 1.3 , 2.46, 0.99, 0.69, 2.38,
       1.1 , 0.81, 1.92, 0.82, 1.05, 0.55, 1.32, 0.88, 0.51, 1.93, 0.36,
       2.53, 2.29, 2.16, 2.88, 0.7 , 2.69, 0.7 , 0.85, 1.22, 0.79],
      dtype=float32)

In [30]:
# Show the candidates re-ranked by the cross-encoder, next to their stage-1 (cosine) rank
order = np.argsort(-sts_scores)                                  # best STS score first
print(" new | old |  STS | cosine | description")
for new_rank, j in enumerate(order, start=1):
    i = candidate_ids[j]
    print(f"{new_rank:4d} | {j + 1:3d} | {sts_scores[j]:4.2f} | {highest_texts[i]:.3f}  | {candidates[j][:90]}")

 new | old |  STS | cosine | description
   1 |   4 | 3.08 | 0.647  | A California businessman is in custody on suspicion of murdering his wife, who has been mi
   2 |  26 | 2.88 | 0.583  | Authorities say Kevin Janson Neal killed his wife late Monday before going on a shooting s
   3 |  28 | 2.69 | 0.581  | A grand jury indicted a 73-year-old man on Thursday for the alleged murder of his first wi
   4 |  23 | 2.53 | 0.586  | Michael Stasko allegedly shot wife and daughter dead before turning gun on himself.
   5 |   8 | 2.46 | 0.632  | Sgt. Michael Walker's girlfriend allegedly stabbed his wife to death in their bedroom.
   6 |   1 | 2.38 | 0.691  | His first wife also died under suspicious circumstances while he was alone with her.
   7 |  11 | 2.38 | 0.625  | The owner of the gun range was killed along with his wife and grandson.
   8 |  24 | 2.29 | 0.585  | Police are searching for the child's father over the woman's death.
   9 |  25 | 2.16 | 0.583  | She left her husband. He kill

In [31]:
# The best pair according to the cross-encoder.
# It is often different from (and more relevant than) the top result of SRoBERTa.
k_similar_posts[int(np.argmax(sts_scores))]
# Original run:
# ['A man killed his wife',
#  'A California businessman is in custody on suspicion of murdering his wife, who has been missing since Wednesday. ...']
# With cosine similarity this post was only 4th.

['A man killed his wife',
 'A California businessman is in custody on suspicion of murdering his wife, who has been missing since Wednesday. The couple']

# Discussion and exercises
1. **Compare the two lists.** Which hits moved up or down after re-ranking? Why might the cosine of a text about a *suicide* be high, while its STS score is low?
2. **Try other posts**, e.g. *"Stock markets fell sharply today"* or *"New vaccine approved for children"*. Re-run from the cell that defines `input_post`.
3. **Change k** (`highest=32`). With k = 5 the re-ranker has little to choose from; with k = 200 it is slower. What happens to the best match?
4. **Swap the embedder** for `sentence-transformers/all-mpnet-base-v2` (`EMBEDDER_NAME`, then re-run Part 3: the old `.npy` file has a different meaning, so delete it first). Do the stage-1 results get better?
5. **Speed.** The search compares the new post with every stored vector. If you normalise all vectors to length 1 once (`normalize_embeddings=True` in `encode`), cosine similarity becomes a plain dot product `post_embeddings @ input_emb`. Measure it with `%%timeit`. For millions of posts, libraries such as FAISS find the nearest vectors without comparing all of them.
6. **Similarity ≠ relevance.** Our cross-encoder was trained on STS-B (*do these two sentences mean the same?*). Search engines often train the re-ranker on *question → answer* data instead. Why could that be better for a question like *"Who won the 2016 election?"*?